# Model Training — AAPL Stock Price Prediction

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.preprocessing import load_and_clean, time_based_split, scale_features
from src.feature_engineering import build_features

In [3]:
DATA_PATH = "../data/raw/AAPL.csv"

df = load_and_clean(DATA_PATH)
df = build_features(df)
df = df.dropna().reset_index(drop=True)

df.head()

,Date,Close,High,Low,Open,Volume,Daily_Return,Log_Return,SMA_10,SMA_20,...,BB_Upper,BB_Lower,Close_lag_1,Close_lag_2,Close_lag_3,Close_lag_5,Close_lag_10,DayOfWeek,Month,Target_Next_Close
0,2010-10-18,9.519469,9.549404,9.408408,9.533539,1093010800,0.010358,0.010305,8.952219,8.761456,...,9.363333,8.159580,9.421876,9.049779,8.984819,8.841727,8.341209,0,10,9.264714
1,2010-10-19,9.264714,9.392838,8.981226,9.082408,1232784000,-0.026761,-0.027126,9.013737,8.799953,...,9.427936,8.171969,9.519469,9.421876,9.049779,8.936921,8.649541,1,10,9.295849
2,2010-10-20,9.295849,9.407209,9.186286,9.250048,721624400,0.003361,0.003355,9.077619,8.834049,...,9.492801,8.175298,9.264714,9.519469,9.421876,8.984819,8.657025,2,10,9.265614
3,2010-10-21,9.265614,9.421876,9.184189,9.350630,551460000,-0.003253,-0.003258,9.138388,8.864883,...,9.544550,8.185216,9.295849,9.264714,9.519469,9.049779,8.657926,3,10,9.204247
4,2010-10-22,9.204247,9.281180,9.169222,9.252143,372778000,-0.006623,-0.006645,9.178502,8.887559,...,9.581305,8.193813,9.265614,9.295849,9.264714,9.421876,8.803111,4,10,9.245257


In [4]:
# Feature columns
target_col = "Target_Next_Close"

feature_cols = [col for col in df.columns
                if col not in ["Date", target_col]]

X_train, X_test, y_train, y_test, train_df, test_df = time_based_split(
    df,
    target_col=target_col,
    feature_cols=feature_cols
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (3249, 25)
Test shape: (574, 25)


In [5]:
X_train_scaled, X_test_scaled, scaler = scale_features(
    X_train,
    X_test
)

print("Scaled Train Shape:", X_train_scaled.shape)
print("Scaled Test Shape:", X_test_scaled.shape)

Scaled Train Shape: (3249, 25)
Scaled Test Shape: (574, 25)


In [6]:
# Train Linear Regression

lr_model = LinearRegression()

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)

print("Linear Regression trained successfully!")

Linear Regression trained successfully!


In [7]:
mae = mean_absolute_error(y_test, lr_pred)
mse = mean_squared_error(y_test, lr_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, lr_pred)

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

MAE  : 2.4056
MSE  : 12.7545
RMSE : 3.5713
R²   : 0.9858


In [8]:
# Train Random Forest

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [9]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print(f"MAE  : {rf_mae:.4f}")
print(f"MSE  : {rf_mse:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"R²   : {rf_r2:.4f}")

MAE  : 28.2220
MSE  : 1441.2514
RMSE : 37.9638
R²   : -0.6051


In [10]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [11]:
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_test, xgb_pred)

print(f"MAE  : {xgb_mae:.4f}")
print(f"MSE  : {xgb_mse:.4f}")
print(f"RMSE : {xgb_rmse:.4f}")
print(f"R²   : {xgb_r2:.4f}")

MAE  : 31.1686
MSE  : 1679.5201
RMSE : 40.9819
R²   : -0.8705


In [12]:
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "MAE": [mae, rf_mae, xgb_mae],
    "RMSE": [rmse, rf_rmse, xgb_rmse],
    "R² Score": [r2, rf_r2, xgb_r2]
})

comparison

,Model,MAE,RMSE,R² Score
0,Linear Regression,2.405550,3.571338,0.985795
1,Random Forest,28.221979,37.963817,-0.605121
2,XGBoost,31.168620,40.981948,-0.870480
